# Pydantic: Data Validation & Settings Management

Pydantic is a Python library for data validation and settings management using Python type annotations. It provides:
- **Type validation** - Automatic type checking and coercion
- **Data serialization** - Convert models to dict/JSON
- **Error handling** - Detailed validation errors
- **Performance** - Built on Rust for speed

## 1. Installation & Basic Model

In [ ]:
# Install pydantic (if needed)
# !pip install pydantic

from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List
from datetime import datetime

# Basic model definition
class User(BaseModel):
    id: int
    name: str
    email: str
    age: Optional[int] = None  # Optional field with default None

# Create and validate data
user = User(id=1, name="Alice", email="alice@example.com", age=30)
print(user)
print(f"\nUser dict: {user.model_dump()}")

## 2. Automatic Type Validation & Coercion

In [ ]:
# Pydantic automatically converts types
user = User(id="42", name="Bob", email="bob@example.com", age="25")
print(f"ID type: {type(user.id)}, value: {user.id}")
print(f"Age type: {type(user.age)}, value: {user.age}")

# Validation errors
try:
    invalid_user = User(id="not_a_number", name="Charlie", email="charlie@example.com")
except ValidationError as e:
    print(f"\nValidation Error:\n{e}")

## 3. Field Validation & Constraints

In [ ]:
from pydantic import field_validator, EmailStr

class Product(BaseModel):
    name: str
    price: float = Field(gt=0, description="Price must be greater than 0")
    quantity: int = Field(ge=0, le=1000, description="Between 0 and 1000")
    
    @field_validator('name')
    @classmethod
    def name_must_not_be_empty(cls, v):
        if len(v.strip()) == 0:
            raise ValueError('Name cannot be empty')
        return v.title()

# Valid product
product = Product(name="laptop", price=999.99, quantity=10)
print(f"Product: {product}\n")

# Invalid price
try:
    invalid = Product(name="phone", price=-50, quantity=5)
except ValidationError as e:
    print(f"Validation Error: {e.errors()[0]['msg']}")

## 4. Nested Models

In [ ]:
class Address(BaseModel):
    street: str
    city: str
    zipcode: str

class Customer(BaseModel):
    name: str
    email: str
    address: Address  # Nested model
    orders: List[str] = []  # List of strings

# Create with nested data
customer_data = {
    "name": "John",
    "email": "john@example.com",
    "address": {
        "street": "123 Main St",
        "city": "New York",
        "zipcode": "10001"
    },
    "orders": ["order_001", "order_002"]
}

customer = Customer(**customer_data)
print(f"Customer: {customer}\n")
print(f"City: {customer.address.city}")

## 5. Model Configuration & Serialization

In [ ]:
import json
from pydantic import ConfigDict

class Config(BaseModel):
    model_config = ConfigDict(
        str_strip_whitespace=True,  # Automatically strip whitespace
        validate_default=True  # Validate default values
    )
    
    database_url: str
    debug: bool = False
    timeout: int = 30

config = Config(database_url="  postgres://localhost  ", debug=True)
print(f"Database URL: '{config.database_url}'")

# Serialization
print(f"\nAs dict: {config.model_dump()}")
print(f"\nAs JSON: {config.model_dump_json(indent=2)}")

## 6. Practical Example: API Request/Response

In [ ]:
from typing import Literal

class BlogPost(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    content: str = Field(min_length=10)
    author: str
    status: Literal["draft", "published", "archived"] = "draft"
    tags: List[str] = []
    created_at: Optional[str] = None

# Simulating API request validation
api_request = {
    "title": "Learning Pydantic",
    "content": "Pydantic is awesome for data validation",
    "author": "Alice",
    "status": "published",
    "tags": ["python", "validation"],
    "created_at": "2024-01-15"
}

post = BlogPost(**api_request)
print(f"Post created: {post.title}")
print(f"Status: {post.status}")
print(f"\nResponse JSON:")
print(post.model_dump_json(indent=2))

## 7. Error Handling Summary

In [ ]:
# Comprehensive error handling example
test_data = {
    "title": "",  # Too short
    "content": "short",  # Too short
    "author": "",
    "status": "invalid"  # Not in allowed values
}

try:
    post = BlogPost(**test_data)
except ValidationError as e:
    print("Validation Errors Found:")
    for error in e.errors():
        field = error['loc'][0]
        message = error['msg']
        print(f"  - {field}: {message}")

## Key Takeaways

✅ **Use Pydantic when:**
- Building APIs (FastAPI uses it)
- Validating user input
- Managing configuration files
- Working with structured data

✅ **Key Features:**
- Type hints drive validation
- Automatic type coercion
- Nested model support
- JSON serialization
- Custom validators
- Detailed error messages